<img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;">

# 11_04 Submission task

In this notebook, you will implement the a whole encoder-decoder Transformer architecture, step by step, from scratch using PyTorch, and you will use it to train a toy machine translation model.    


## Exercise 0 — Bring your building blocks

In the code cell below, copy/paste all the core modules you implemented in the previous three practice notebooks. We’ll reuse them to assemble a full seq2seq Transformer.

In [1]:
import math
import torch
import torch.nn as nn

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.shape[1], :]

class LearnedPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)
        nn.init.normal_(self.pos_embedding.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, _ = x.shape
        positions = torch.arange(0, T, device=x.device)
        return x + self.pos_embedding(positions)

class AddNorm(nn.Module):
    def __init__(self, d_model, pre_norm):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pre_norm = pre_norm

    def forward(self, x, sublayer):
        if self.pre_norm:
            return x + sublayer(self.norm(x))
        else:
            return self.norm(x + sublayer(x))

class FFN(nn.Module):
    def __init__(self, d_model, d_ff, activation="gelu"):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

        if activation.lower() == "relu":
            self.activation = nn.ReLU()
        elif activation.lower() == "gelu":
            self.activation = nn.GELU()
        else:
            raise ValueError(f"Unsupported activation: {activation}")

    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x)))

## Exercise 1 — Verify `MultiHeadAttention` supports additive masks

Your `MultiHeadAttention` must accept an additive attention mask (`attn_mask`) that is broadcastable to shape `(B, h, T_q, T_k)` and is 0 for keep and -inf for mask. The mask is **added** to the attention scores *before* softmax. The mask will be used for masked multi-head attention and for handling padding masks.

**Checklist**
- Forward signature: `forward(x_q, x_kv, attn_mask=None)`
- Compute scores as: `scores = (Q @ K^T) / sqrt(d_k)`
- If `attn_mask` is not `None`: `scores = scores + attn_mask`
- Then `softmax(scores, dim=-1)`

If your implementation already does this, you are done with exercise 1.


In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_k
        self.d_v = d_v

        self.W_Q = nn.Linear(d_model, num_heads * d_k, bias=False)
        self.W_K = nn.Linear(d_model, num_heads * d_k, bias=False)
        self.W_V = nn.Linear(d_model, num_heads * d_v, bias=False)
        self.W_o = nn.Linear(num_heads * d_v, d_model, bias=False)

        self.scale = math.sqrt(d_k)

    def forward(self, x_q, x_kv, attn_mask=None):
        B, T_q, _ = x_q.shape
        _, T_k, _ = x_kv.shape

        Q_all = self.W_Q(x_q)
        K_all = self.W_K(x_kv)
        V_all = self.W_V(x_kv)

        Q = Q_all.view(B, T_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K_all.view(B, T_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V_all.view(B, T_k, self.num_heads, self.d_v).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / self.scale

        if attn_mask is not None:
            scores = scores + attn_mask

        attention_weights = torch.softmax(scores, dim=-1)
        out_heads = attention_weights @ V
        out_joined = out_heads.permute(0, 2, 1, 3).flatten(start_dim=2)

        return self.W_o(out_joined)

## Exercise 2 — Thread the encoder padding mask through `EncoderLayer` → `Encoder`

Pass an **encoder padding mask** (additive) from the `Encoder.forward(attn_mask=...)` into each `EncoderLayer`, and then into the self-attention.

**Tasks**
1. `Encoder.forward(x, attn_mask=None)` must pass `attn_mask` into each layer.
2. `EncoderLayer.forward(x, attn_mask=None)` must pass `attn_mask` into its self-attention.
3. Inside attention, **add** the mask to scores before softmax.

**Tip**  
The simplest encoder PAD mask is shaped `(B, 1, 1, T_src)` with `-inf` where PAD and `0` elsewhere; it will broadcast across heads and queries.


In [3]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads, d_ff, prenorm, activation="gelu"):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, d_k, d_v, num_heads)
        self.add_norm_1 = AddNorm(d_model, pre_norm=prenorm)
        self.ffn = FFN(d_model, d_ff, activation)
        self.add_norm_2 = AddNorm(d_model, pre_norm=prenorm)

    def forward(self, x, attn_mask=None):
        x = self.add_norm_1(x, lambda x_: self.self_attn(x_q=x_, x_kv=x_, attn_mask=attn_mask))
        x = self.add_norm_2(x, self.ffn)
        return x


class Encoder(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads, num_layers, d_ff, max_len=512, prenorm=True, activation="gelu", positional_encoding_type="sinusoidal"):
        super().__init__()
        if positional_encoding_type == "sinusoidal":
            self.pos_encoder = SinusoidalPositionalEncoding(max_len=max_len, d_model=d_model)
        elif positional_encoding_type == "learned":
            self.pos_encoder = LearnedPositionalEncoding(max_len=max_len, d_model=d_model)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, d_k, d_v, num_heads, d_ff, prenorm, activation)
            for _ in range(num_layers)
        ])
        self.prenorm = prenorm
        if self.prenorm:
            self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask=None):
        x = self.pos_encoder(x)
        for layer in self.layers:
            x = layer(x, attn_mask=attn_mask)
        if self.prenorm:
            x = self.final_norm(x)
        return x

## Exercise 3 — Complete the `Decoder` masks

Your `Decoder` must:
1) Build a causal mask `(B, h, T_dec, T_dec)` with `-inf` above the main diagonal.  
2) Optionally add target PAD mask `(B, h, 1, T_dec)` (mask columns/keys at PAD).  
3) Build cross-attention PAD mask from encoder IDs `(B, h, 1, T_enc)` and broadcast it in cross-attn.  

Then pass `self_mask` and `cross_mask` into every `DecoderLayer`.


In [4]:
def make_causal_mask(B: int, num_heads: int, T_dec: int, device=None) -> torch.Tensor:
    mask = torch.triu(torch.full((T_dec, T_dec), float('-inf'), device=device), diagonal=1)
    causal_mask = mask.unsqueeze(0).unsqueeze(0).expand(B, num_heads, T_dec, T_dec)
    
    return causal_mask

class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads, d_ff, prenorm, activation="gelu"):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, d_k, d_v, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, d_k, d_v, num_heads)
        self.ffn = FFN(d_model, d_ff, activation)

        self.add_norm_1 = AddNorm(d_model, pre_norm=prenorm)
        self.add_norm_2 = AddNorm(d_model, pre_norm=prenorm)
        self.add_norm_3 = AddNorm(d_model, pre_norm=prenorm)

    def forward(self, x, enc_out, self_mask=None, cross_mask=None):
        x = self.add_norm_1(x, lambda x_: self.self_attn(x_q=x_, x_kv=x_, attn_mask=self_mask))
        x = self.add_norm_2(x, lambda x_: self.cross_attn(x_q=x_, x_kv=enc_out, attn_mask=cross_mask))
        x = self.add_norm_3(x, self.ffn)
        return x


class Decoder(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads, num_layers, d_ff, max_len=512, prenorm=True, activation="gelu", positional_encoding_type="sinusoidal"):
        super().__init__()
        if positional_encoding_type == "sinusoidal":
            self.pos_encoder = SinusoidalPositionalEncoding(max_len=max_len, d_model=d_model)
        elif positional_encoding_type == "learned":
            self.pos_encoder = LearnedPositionalEncoding(max_len=max_len, d_model=d_model)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, d_k, d_v, num_heads, d_ff, prenorm, activation)
            for _ in range(num_layers)
        ])
        self.prenorm = prenorm
        if self.prenorm:
            self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x_dec, enc_out, self_pad_mask=None, enc_pad_mask=None):
        B, T_dec, _ = x_dec.shape
        num_heads = self.layers[0].self_attn.num_heads

        x = self.pos_encoder(x_dec)

        causal_mask = make_causal_mask(B, self.layers[0].self_attn.num_heads, T_dec, device=x.device)

        if self_pad_mask is not None:
            self_mask = causal_mask + self_pad_mask
        else:
            self_mask = causal_mask

        cross_mask = enc_pad_mask

        for layer in self.layers:
            x = layer(x, enc_out, self_mask=self_mask, cross_mask=cross_mask)

        if self.prenorm:
            x = self.final_norm(x)

        return x


class DecoderHead(nn.Module):
    def __init__(self, d_model, vocab_size, bias=True):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size, bias=bias)

    def forward(self, h_dec):
        return self.proj(h_dec)

## Exercise 4 — Implement `Seq2Seq`

Wire together:
- Source embedding
- Encoder (pass encoder PAD mask)
- Target embedding (+ scale)
- Decoder (will build causal + use PAD masks you pass)
- DecoderHead (linear → logits; optional weight tying to target embedding)

**Inputs**
- `src_ids: (B, T_src)` token IDs
- `tgt_ids: (B, T_tgt)` token IDs (teacher forcing input; typically includes BOS)

**Forward**
1. Embed `src_ids`, run **Encoder** with an **encoder PAD mask**:
   - Mask shape `(B,1,1,T_src)` with `-inf` at PAD; broadcast inside attention to `(B,h,*,T_src)`.
2. Embed `tgt_ids`, run **Decoder** passing:
   - `self_pad_mask_ids=tgt_ids`
   - `enc_pad_mask_ids=src_ids`
3. Project decoder outputs to logits: `(B, T_tgt, V)`.

Return `logits`.


In [5]:
class Seq2Seq(nn.Module):
    def __init__(self,src_vocab_size,tgt_vocab_size,d_model=64,num_layers_enc=2,num_layers_dec=2,num_heads=4,d_ff=128,pad_id=0,max_len=512):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        d_k = d_v = d_model // num_heads

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.scale = math.sqrt(d_model)

        self.encoder = Encoder(d_model, d_k, d_v, num_heads, num_layers_enc, d_ff, max_len=max_len)
        self.decoder = Decoder(d_model, d_k, d_v, num_heads, num_layers_dec, d_ff, max_len=max_len)
        self.head = DecoderHead(d_model, tgt_vocab_size)

    def make_pad_mask(self, seq_ids):
        mask = (seq_ids == self.pad_id).unsqueeze(1).unsqueeze(2)
        return mask.masked_fill(mask, float('-inf')).masked_fill(~mask, 0.0)

    def forward(self, src_ids, tgt_ids):
        enc_pad_mask = self.make_pad_mask(src_ids)
        tgt_pad_mask = self.make_pad_mask(tgt_ids)

        src_emb = self.src_embedding(src_ids) * self.scale
        enc_out = self.encoder(src_emb, attn_mask=enc_pad_mask)

        tgt_emb = self.tgt_embedding(tgt_ids) * self.scale
        dec_out = self.decoder(
            tgt_emb, 
            enc_out, 
            self_pad_mask=tgt_pad_mask, 
            enc_pad_mask=enc_pad_mask
        )

        logits = self.head(dec_out)
        return logits

## Exercise 5 — Implement one training step (Teacher Forcing)

In this exercise, you will implement a function that performs one training step for your seq2seq Transformer model.  
This means taking one batch of source–target sentence pairs, running them through the model, computing the loss, and updating the model parameters.

### Goal

You must implement a function:

```python
def training_step(model, batch, optimizer, pad_id, device=None, grad_clip=1.0):
    ...
```

This function performs one gradient update on a single batch.

### Inputs

| Parameter   | Type                       | Description                                                         |
| ----------- | -------------------------- | ------------------------------------------------------------------- |
| `model`     | `nn.Module`                | Your seq2seq Transformer model (`Seq2Seq` class).                   |
| `batch`     | tuple `(src_ids, tgt_ids)` | A minibatch of padded source and target token IDs.                  |
| `optimizer` | `torch.optim.Optimizer`    | The optimizer used to update parameters (e.g. AdamW).               |
| `pad_id`    | `int`                      | ID of the padding token. Loss should ignore positions with this ID. |
| `device`    | `torch.device`             | Device to move tensors to (`"cuda"` or `"cpu"`).                    |
| `grad_clip` | `float`                    | Maximum gradient norm for clipping (avoid exploding gradients).     |

### Expected shapes

| Tensor    | Shape               | Description                                                              |
| --------- | ------------------- | ------------------------------------------------------------------------ |
| `src_ids` | `(B, T_src)`        | Source sentence token IDs (e.g. in English).                             |
| `tgt_ids` | `(B, T_tgt)`        | Target sentence token IDs (e.g. in Spanish).                             |
| `logits`  | `(B, T_tgt - 1, V)` | Model output: vocabulary logits for each decoder position (except last). |
| `tgt_out` | `(B, T_tgt - 1)`    | Target labels: the next token after each decoder input step.             |

### Step-by-step description

1. Move the input tensors to the target device
   * Move both the source (src_ids) and target (tgt_ids) tensors to the same device as the model
  
2. Prepare decoder inputs and outputs (Teacher Forcing)

   * During training, the decoder receives the true previous tokens as input, not its own predictions.
   * We therefore shift the target sequence one position:
       * `tgt_in`: all tokens except the last one (input to the decoder)
       * `tgt_out`: all tokens except the first one (expected outputs)
| Full target (`tgt_ids`) | Decoder input (`tgt_in`) | Expected output (`tgt_out`) |
|--------------------------|---------------------------|------------------------------|
| `<bos> me gusta el café <eos>` | `<bos> me gusta el café` | `me gusta el café <eos>` |

3. Forward pass through the model
   * Call your seq2seq model with the source and decoder input tokens.
   * The output logits has shape `(B, T_tgt - 1, V)`.
4. Compute the cross entropy loss
   * The goal is to minimize the difference between model predictions and ground truth `tgt_out`.
   * Since CrossEntropyLoss expects (N, C) and (N,) shapes, you must flatten both T and batch dimensions.
   * Make sure that padding positions (pad_id) are ignored by the loss (you can use the `ignore_index` parameter).
5. Backward pass and optimization
   * Standard PyTorch training pattern.
6. Return the loss value
   * Return a scalar loss (Python float) for logging.

In [6]:
def training_step(model, batch, optimizer, pad_id, device=None, grad_clip=1.0):
    model.train()
    
    src_ids, tgt_ids = batch
    if device is not None:
        src_ids = src_ids.to(device)
        tgt_ids = tgt_ids.to(device)

    tgt_in = tgt_ids[:, :-1]   
    tgt_out = tgt_ids[:, 1:]   

    optimizer.zero_grad()
    logits = model(src_ids, tgt_in)

    vocab_size = logits.shape[-1]
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id)
    loss = loss_fn(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))

    loss.backward()
    if grad_clip > 0:
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    return loss.item()

## Exercise 6 — Implement Greedy Decoding (Inference)

Now that your model is ready to be trained to predict the next token given previous ones,  
you will implement greedy decoding, the simplest inference strategy.

Greedy decoding means that at each time step, you pick the token with the highest probability from the model’s output, 
and feed it back into the decoder for the next prediction.

### Goal

Implement a function:

```python
def greedy_decode(model, src_ids, max_len, bos_id, eos_id, pad_id, device):
    ...
```
that generates an output sequence (e.g., Spanish translation) from an input sentence (e.g., English source).

### Input

| Parameter | Type                | Description                                      |
| --------- | ------------------- | ------------------------------------------------ |
| `model`   | `nn.Module`         | Your seq2seq Transformer model (trained).        |
| `src_ids` | `(1, T_src)` tensor | Source sentence token IDs (batch size = 1).      |
| `max_len` | `int`               | Maximum length of the generated target sequence. |
| `bos_id`  | `int`               | Beginning-of-sentence token ID.                  |
| `eos_id`  | `int`               | End-of-sentence token ID.                        |
| `pad_id`  | `int`               | Padding token ID (for masks).                    |
| `device`  | `torch.device`      | Device on which to perform decoding.             |


### Step-by-step description

1. Encode the source
2. Initizalize the decoder input
3. Iteratively generate tokens
   * Loop up to `max_len` times:
       * Pass the tokens generated so far into the decoder
       * Get the output logits from the model
       * Take the logits from the *last step*.
       * Apply softmax and the the argmax
4. Return the generated sequence

In [7]:
def greedy_decode(model, src_ids, max_len, bos_id, eos_id, pad_id, device):
    model.eval()
    with torch.no_grad():
        if device is not None:
            src_ids = src_ids.to(device)

        enc_pad_mask = model.make_pad_mask(src_ids)
        src_emb = model.src_embedding(src_ids) * model.scale
        enc_out = model.encoder(src_emb, attn_mask=enc_pad_mask)

        ys = torch.ones(1, 1, dtype=torch.long, device=device).fill_(bos_id)

        for _ in range(max_len - 1):
            tgt_pad_mask = model.make_pad_mask(ys)
            tgt_emb = model.tgt_embedding(ys) * model.scale
            dec_out = model.decoder(tgt_emb, enc_out, self_pad_mask=tgt_pad_mask, enc_pad_mask=enc_pad_mask)
            
            logits = model.head(dec_out[:, -1, :])
            next_word = torch.argmax(logits, dim=-1).item()

            ys = torch.cat([ys, torch.ones(1, 1, dtype=torch.long, device=device).fill_(next_word)], dim=1)

            if next_word == eos_id:
                break

        return ys.squeeze(0)

## Exercise 7 — Mini English→Spanish Toy Training

Now that you have all components of your Transformer (encoder, decoder, masking, and training loop),
you’ll train a very small model on a handcrafted English–Spanish dataset.

The goal here is *not* to achieve great performance, but to show that your model:
- runs end-to-end without errors,
- the loss decreases over epochs,
- and it starts learning simple word translations.

### 1. Dataset

We’ll define a tiny parallel corpus of just a few English–Spanish sentence pairs, for example:

| English | Spanish |
|----------|----------|
| “i am hungry” | “tengo hambre” |
| “he is tired” | “esta cansado” |
| “she likes pizza” | “le gusta la pizza” |
| “the cat sleeps” | “el gato duerme” |

This data will be tokenized *by words* (no subword model or external tokenizer needed).

### 2. Tokenization

We’ll build simple vocabularies from the dataset itself.

Each word will have an integer ID:
```python
src_vocab = { "<pad>":0, "<bos>":1, "<eos>":2, "i":3, "am":4, "hungry":5, ... }
tgt_vocab = { "<pad>":0, "<bos>":1, "<eos>":2, "tengo":3, "hambre":4, ... }
```

### 3. Model setup

We’ll use your `Seq2Seq` Transformer with smaller dimensions:
```python
model = Seq2Seq(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=64,
    num_layers_enc=2,
    num_layers_dec=2,
    num_heads=4,
    d_ff=128,
    pad_id=PAD
)
```

### 4. Training and predicting

1. Carry out the training loop, and train your model for a few epochs. Print the loss.
2. Use greedy decoding to obtain the translation. Print a few translated sentences

In [8]:
raw_data = [
    ("i am hungry", "tengo hambre"),
    ("he is tired", "esta cansado"),
    ("she likes pizza", "le gusta la pizza"),
    ("the cat sleeps", "el gato duerme")
]

PAD_ID, BOS_ID, EOS_ID = 0, 1, 2

src_words = set(w for src, _ in raw_data for w in src.split())
tgt_words = set(w for _, tgt in raw_data for w in tgt.split())

src_vocab = {"<pad>": PAD_ID, "<bos>": BOS_ID, "<eos>": EOS_ID, **{w: i+3 for i, w in enumerate(src_words)}}
tgt_vocab = {"<pad>": PAD_ID, "<bos>": BOS_ID, "<eos>": EOS_ID, **{w: i+3 for i, w in enumerate(tgt_words)}}

inv_tgt_vocab = {v: k for k, v in tgt_vocab.items()}

def encode(seq, vocab):
    return [BOS_ID] + [vocab[w] for w in seq.split()] + [EOS_ID]

def pad_sequence(seqs, pad_id=0):
    max_len = max(len(s) for s in seqs)
    return torch.tensor([s + [pad_id] * (max_len - len(s)) for s in seqs])

src_batch = pad_sequence([encode(src, src_vocab) for src, _ in raw_data])
tgt_batch = pad_sequence([encode(tgt, tgt_vocab) for _, tgt in raw_data])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Seq2Seq(src_vocab_size=len(src_vocab),tgt_vocab_size=len(tgt_vocab),d_model=64,num_layers_enc=2,num_layers_dec=2,num_heads=4,d_ff=128,pad_id=PAD_ID).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("--- Starting Training ---")
for epoch in range(1, 101):
    loss = training_step(model, (src_batch, tgt_batch), optimizer, pad_id=PAD_ID, device=device)
    if epoch % 30 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f}")

print("\n--- Translation Results ---")
for src_str, _ in raw_data:
    src_tokens = torch.tensor([encode(src_str, src_vocab)], device=device)
    pred_ids = greedy_decode(model, src_tokens, max_len=10, bos_id=BOS_ID, eos_id=EOS_ID, pad_id=PAD_ID, device=device)
    
    pred_words = [inv_tgt_vocab[i.item()] for i in pred_ids if i.item() not in (PAD_ID, BOS_ID, EOS_ID)]
    print(f"Input: {src_str:18s} -> Translation: {' '.join(pred_words)}")
    

--- Starting Training ---
Epoch   1 | Loss: 2.9569
Epoch  30 | Loss: 0.6204
Epoch  60 | Loss: 0.0469
Epoch  90 | Loss: 0.0197

--- Translation Results ---
Input: i am hungry        -> Translation: tengo hambre
Input: he is tired        -> Translation: esta cansado
Input: she likes pizza    -> Translation: le gusta la pizza
Input: the cat sleeps     -> Translation: el gato duerme
